# VoxDefend: LCNN-9 Anti-Spoofing Training on Google Colab

**Project**: VoxDefend (VocalShield AI)
**Model Architecture**: Light-CNN-9 (LCNN-9) with Max-Feature-Map (MFM) Activation
**Input**: 16 kHz mono audio, exactly 48,000 samples (3.0s) -> Log-Mel Spectrogram `[B, 1, 80, 301]`
**Output**: 1 raw logit -> Platt Scaling calibrated probability $P(\text{spoof}) \in [0, 1]$
**Dataset**: ASVspoof 2019 Logical Access (LA) Partition (Official Edinburgh DataShare)
**Evaluation Standards**: Equal Error Rate (EER), APCER (FAR), BPCER (FRR), ROC-AUC
**Export Target**: ONNX Opset 17 with ONNX Runtime parity verification

## 1. Google Drive Mount & Hardware Verification
Mounts Google Drive for persistent checkpoint storage to survive Colab session disconnections.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Verify GPU Acceleration
!nvidia-smi

import torch
assert torch.cuda.is_available(), "CUDA GPU is required for Colab training! Go to Runtime -> Change runtime type -> T4 GPU."
print(f"[OK] CUDA Available: {torch.cuda.get_device_name(0)}")
print(f"[OK] VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Environment Setup & Dependency Installation

In [ ]:
!pip install --upgrade pip
!pip install torch torchaudio onnx onnxruntime scikit-learn scipy matplotlib

# Set project directory
import os
from pathlib import Path

# If running directly from cloned repo:
REPO_PATH = Path('/content/VOCALSHIELD-AI')
if not REPO_PATH.exists():
    # Clone or symlink
    print("Please upload or clone your repository to /content/VOCALSHIELD-AI")
os.chdir(str(REPO_PATH))
print(f"Current working directory: {os.getcwd()}")

## 3. Dataset Acquisition (ASVspoof 2019 LA)
Downloads or links the official ASVspoof 2019 LA dataset from Edinburgh DataShare.
Optionally copies from Google Drive if already downloaded.

In [ ]:
# Path to data directory on Colab fast local SSD
DATA_DIR = Path('/content/data/asvspoof2019_la')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Check if LA.zip exists in Google Drive
DRIVE_ZIP = Path('/content/drive/MyDrive/asvspoof2019/LA.zip')
LOCAL_ZIP = DATA_DIR / 'LA.zip'

if DRIVE_ZIP.exists():
    print(f"Found LA.zip in Google Drive: {DRIVE_ZIP}. Copying to local SSD...")
    !cp "{DRIVE_ZIP}" "{LOCAL_ZIP}"
elif not (DATA_DIR / 'ASVspoof2019_LA_train').exists():
    print("Downloading official ASVspoof 2019 LA dataset from Edinburgh DataShare...")
    !wget -O "{LOCAL_ZIP}" "https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip?sequence=3&isAllowed=y"

# Unpack if needed
if LOCAL_ZIP.exists() and not (DATA_DIR / 'ASVspoof2019_LA_train').exists():
    print("Extracting LA.zip...")
    !unzip -q "{LOCAL_ZIP}" -d "{DATA_DIR}"
    print("[OK] Dataset extracted successfully.")

## 4. Pre-Training Offline Verification Gate
Executes the nine mandatory offline checks to verify configuration, protocol hashes, dataset loader, tensor shapes, loss gradients, EER metrics, Platt calibration, and ONNX parity before any training begins.

In [ ]:
# Run the 9 pre-training verification checks
!python scripts/verify_training_prep.py

## 5. Model Training with Google Drive Checkpoint Persistence
Runs the VoxDefend training loop with:
- AdamW optimizer & CosineAnnealingLR
- BCEWithLogitsLoss with pos_weight ~8.84
- Training-only augmentations
- Deterministic validation on Dev partition
- Checkpoint saving to Google Drive (`/content/drive/MyDrive/VoxDefend/checkpoints`)
- Early stopping on `val_eer` (patience=7, min_delta=0.0005)

In [ ]:
# Checkpoint directory in Google Drive
CHECKPOINT_DIR = "/content/drive/MyDrive/VoxDefend/checkpoints"
DATA_PATH = "/content/data/asvspoof2019_la"

!python -m ml.train \
    --config training_config.json \
    --data_dir "{DATA_PATH}" \
    --checkpoint_dir "{CHECKPOINT_DIR}" \
    --device cuda

## 6. Resume from Checkpoint (if session was interrupted)
If the Colab session was disconnected or timed out, run this command to seamlessly resume training from the latest checkpoint stored on Google Drive.

In [ ]:
# Resume command
!python -m ml.train \
    --config training_config.json \
    --data_dir "{DATA_PATH}" \
    --checkpoint_dir "{CHECKPOINT_DIR}" \
    --resume "{CHECKPOINT_DIR}/latest_checkpoint.pt" \
    --device cuda

## 7. Inspection of Platt Calibration & Empirical Operational Zones
Loads the fitted `platt_calibrator.json` and visualizes the calibration mapping and operational decision zones.

In [ ]:
import json
from pathlib import Path
from ml.calibration.platt import PlattCalibrator

cal_file = Path(CHECKPOINT_DIR) / "platt_calibrator.json"
if cal_file.exists():
    calibrator = PlattCalibrator.load(cal_file)
    print("--- Platt Calibration Status ---")
    print(f"Scaling Weight (w):    {calibrator.w:.4f}")
    print(f"Scaling Bias (b):      {calibrator.b:.4f}")
    print(f"Dev Set EER:           {calibrator.dev_eer * 100:.2f}%")
    print(f"EER Threshold:         {calibrator.theta_eer:.4f}")
    print(f"Human Zone Threshold:  {calibrator.theta_human:.4f} (FAR <= {calibrator.target_far*100:.1f}%)")
    print(f"Spoof Zone Threshold:  {calibrator.theta_synthetic:.4f} (FRR <= {calibrator.target_frr*100:.1f}%)")
    print("\nOperational Zones:")
    print(f"  Bona Fide / Human:  [0.0000, {calibrator.theta_human:.4f}]")
    print(f"  Uncertain (Step-Up): ({calibrator.theta_human:.4f}, {calibrator.theta_synthetic:.4f})")
    print(f"  Synthetic / Spoof:  [{calibrator.theta_synthetic:.4f}, 1.0000]")
else:
    print(f"Calibrator file not found at {cal_file}. Run training first.")

## 8. ONNX Export & Runtime Verification
Verifies the exported `voxdefend_lcnn9.onnx` model (opset 17) against ONNX Runtime.

In [ ]:
import onnx
import onnxruntime as ort
import numpy as np

onnx_path = Path(CHECKPOINT_DIR) / "voxdefend_lcnn9.onnx"
if onnx_path.exists():
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)
    print(f"[OK] ONNX model valid: {onnx_path}")
    
    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    dummy_spec = np.random.randn(1, 1, 80, 301).astype(np.float32)
    ort_out = session.run(None, {"input_spectrogram": dummy_spec})[0]
    print(f"[OK] ONNX Runtime execution successful. Logit output shape: {ort_out.shape}, value: {ort_out[0, 0]:.4f}")
else:
    print(f"ONNX model not found at {onnx_path}.")